# PyTorch vs Keras — AI Engineering Interview Prep

Same MLP on Iris dataset implemented in both frameworks side-by-side.

In [ ]:
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

np.random.seed(42)

iris = load_iris()
X, y = iris.data.astype(np.float32), iris.target
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

scaler = StandardScaler()
X_tr = scaler.fit_transform(X_tr).astype(np.float32)
X_te = scaler.transform(X_te).astype(np.float32)

print(f"Train: {X_tr.shape}, Test: {X_te.shape}")

## 1. PyTorch Implementation

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import time

torch.manual_seed(42)

# Data
Xtr_t = torch.tensor(X_tr)
ytr_t = torch.tensor(y_tr, dtype=torch.long)
Xte_t = torch.tensor(X_te)
yte_t = torch.tensor(y_te, dtype=torch.long)

train_dl = DataLoader(TensorDataset(Xtr_t, ytr_t), batch_size=16, shuffle=True)

# Model
class MLP_PyTorch(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(4, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 3)
        )
    def forward(self, x):
        return self.net(x)

model_pt = MLP_PyTorch()
optimizer = optim.Adam(model_pt.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

# Training loop
start = time.perf_counter()
pt_losses = []
for epoch in range(100):
    model_pt.train()
    for xb, yb in train_dl:
        optimizer.zero_grad()
        loss = criterion(model_pt(xb), yb)
        loss.backward()
        optimizer.step()
    model_pt.eval()
    with torch.no_grad():
        pt_losses.append(criterion(model_pt(Xtr_t), ytr_t).item())

model_pt.eval()
with torch.no_grad():
    pt_acc = (model_pt(Xte_t).argmax(1) == yte_t).float().mean().item()

pt_time = time.perf_counter() - start
print(f"PyTorch — test acc: {pt_acc:.4f}, time: {pt_time:.2f}s")

## 2. Keras/TensorFlow Implementation

In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

try:
    import tensorflow as tf
    from tensorflow import keras

    tf.random.set_seed(42)

    model_keras = keras.Sequential([
        keras.layers.Dense(32, activation='relu', input_shape=(4,)),
        keras.layers.Dropout(0.2),
        keras.layers.Dense(16, activation='relu'),
        keras.layers.Dense(3, activation='softmax')
    ])

    model_keras.compile(
        optimizer=keras.optimizers.Adam(1e-3),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    start = time.perf_counter()
    history = model_keras.fit(
        X_tr, y_tr, epochs=100, batch_size=16,
        validation_data=(X_te, y_te), verbose=0
    )
    keras_time = time.perf_counter() - start

    _, keras_acc = model_keras.evaluate(X_te, y_te, verbose=0)
    keras_losses = history.history['loss']
    print(f"Keras — test acc: {keras_acc:.4f}, time: {keras_time:.2f}s")
    keras_available = True
except ImportError:
    print("TensorFlow not installed. Install with: pip install tensorflow")
    keras_available = False
    keras_losses = [1.0] * 100

## 3. Comparison

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 4))
plt.plot(pt_losses, label='PyTorch loss')
plt.plot(keras_losses, label='Keras loss')
plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.title('Training Loss Comparison')
plt.legend()
plt.show()

print("\n=== Framework Comparison ===")
comparison = {
    'PyTorch':         {'style': 'Imperative (define-by-run)', 'debug': 'Native Python debugger', 'deploy': 'TorchScript/ONNX', 'research': 'Dominant'},
    'Keras/TF':        {'style': 'Declarative (compile+fit)', 'debug': 'Eager mode / tfdbg', 'deploy': 'TFServing/TFLite', 'research': 'Common'},
}
import pandas as pd
pd.DataFrame(comparison).T

## Interview Tips

- **PyTorch**: dynamic computation graph (define-by-run). Easier debugging, dominant in research.
- **Keras**: high-level API. `.compile()/.fit()` is great for prototyping; less flexible for custom loops.
- **Both support**: ONNX export, GPU acceleration, distributed training, custom layers.
- **JAX/Flax**: functional style, JIT compilation, emerging in research (Google DeepMind).
- **When to choose**: PyTorch for research/custom architectures; Keras for quick prototyping; TF for production with TFServing.